1. Revenue by Customer

In [0]:
%python
from pyspark.sql.functions import *

silver_orders   = spark.read.table("streaming_project.ecommerce.silver_orders")
silver_products = spark.read.table("streaming_project.ecommerce.silver_products")
silver_customers = spark.read.table("streaming_project.ecommerce.silver_customers")

# Join orders with products to get price
orders_with_price = (
    silver_orders
    .join(silver_products.select("product_id", "price", "category"), on="product_id", how="left")
    .withColumn("line_total", col("quantity") * col("price"))
)

# Revenue per customer
gold_customer_revenue = (
    orders_with_price
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_items_bought"),
        round(sum("line_total"), 2).alias("total_revenue"),
        round(avg("line_total"), 2).alias("avg_order_value"),
        max("order_date").alias("last_order_date")
    )
    .join(silver_customers.select("customer_id", "full_name", "email"), on="customer_id", how="left")
    .orderBy(col("total_revenue").desc())
)

(
    gold_customer_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("streaming_project.ecommerce.gold_customer_revenue")
)

print("✅ gold_customer_revenue written")
display(spark.read.table("streaming_project.ecommerce.gold_customer_revenue"))

✅ gold_customer_revenue written


customer_id,total_orders,total_items_bought,total_revenue,avg_order_value,last_order_date,full_name,email
1,2,27,3376.74,562.79,2020-03-02T00:00:00.000Z,John Doe,john@gmail.com
4,1,5,560.00,280.00,2020-03-01T00:00:00.000Z,Don Romer,don@gmail.com
3,2,6,460.78,153.59,2020-03-01T00:00:00.000Z,Kevin Ryan,kevin@gmail.com
2,1,3,283.90,141.95,2020-03-01T00:00:00.000Z,David Morrison,morrison@gmail.com
8,1,1,9.85,9.85,2020-03-01T00:00:00.000Z,William Hopkins,william@gmail.com


2. Popular Products

In [0]:
%python
from pyspark.sql.functions import *

silver_orders   = spark.read.table("streaming_project.ecommerce.silver_orders")
silver_products = spark.read.table("streaming_project.ecommerce.silver_products")

gold_popular_products = (
    silver_orders
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .join(
        silver_products.select("product_id", "title", "category", "price", "rating_rate", "price_category"),
        on="product_id", how="left"
    )
    .withColumn("total_revenue", round(col("total_quantity_sold") * col("price"), 2))
    .orderBy(col("total_quantity_sold").desc())
)

(
    gold_popular_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("streaming_project.ecommerce.gold_popular_products")
)

print("✅ gold_popular_products written")
display(spark.read.table("streaming_project.ecommerce.gold_popular_products"))

✅ gold_popular_products written


product_id,total_quantity_sold,total_orders,unique_customers,title,category,price,rating_rate,price_category,total_revenue
1,20,4,3,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops",Men's Clothing,109.95,3.9,Premium,2199.00
3,6,1,1,Mens Cotton Jacket,Men's Clothing,55.99,4.7,Mid-Range,335.94
2,5,2,1,Mens Casual Premium Slim Fit T-Shirts,Men's Clothing,22.30,4.1,Mid-Range,111.50
12,3,1,1,WD 4TB Gaming Drive Works with Playstation 4 Portable External Hard Drive,Electronics,114.00,4.8,Premium,342.00
5,2,1,1,John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet,Jewelery,695.00,4.6,Premium,1390.00
10,2,1,1,SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s,Electronics,109.00,2.9,Premium,218.00
9,1,1,1,WD 2TB Elements Portable External Hard Drive - USB 3.0,Electronics,64.00,3.3,Mid-Range,64.00
7,1,1,1,White Gold Plated Princess,Jewelery,9.99,3.0,Budget,9.99
8,1,1,1,Pierced Owl Rose Gold Plated Stainless Steel Double,Jewelery,10.99,1.9,Budget,10.99
18,1,1,1,MBJ Women's Solid Short Sleeve Boat Neck V,Women's Clothing,9.85,4.7,Budget,9.85


3. Revenue by Category & Month

In [0]:
%python
from pyspark.sql.functions import *

silver_orders   = spark.read.table("streaming_project.ecommerce.silver_orders")
silver_products = spark.read.table("streaming_project.ecommerce.silver_products")

gold_category_revenue = (
    silver_orders
    .join(silver_products.select("product_id", "price", "category"), on="product_id", how="left")
    .withColumn("line_total", col("quantity") * col("price"))
    .groupBy("category", "order_year", "order_month")
    .agg(
        sum("line_total").alias("total_revenue"),
        sum("quantity").alias("total_units_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers")
    )
    .orderBy("category", "order_year", "order_month")
)

(
    gold_category_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("streaming_project.ecommerce.gold_category_revenue")
)

print("✅ gold_category_revenue written")
display(spark.read.table("streaming_project.ecommerce.gold_category_revenue"))

✅ gold_category_revenue written


category,order_year,order_month,total_revenue,total_units_sold,total_orders,unique_customers
Electronics,2020,3,624.00,6,2,2
Jewelery,2020,1,1390.00,2,1,1
Jewelery,2020,3,20.98,2,1,1
Men's Clothing,2020,1,1628.50,18,2,2
Men's Clothing,2020,3,1017.94,13,2,2
Women's Clothing,2020,3,9.85,1,1,1
